In [1]:
import os
os.chdir('..')

In [ ]:
from src.data.datasets.cwru_dataset import CWRUDataset
from src.models.architecture.mobilenet_v4 import mobilenetv4
from src.models.architecture.ghostnet_v3 import ghostnetv3
from src.models.architecture.edgenext_xxs import edgenext_xxs
from src.models.architecture.tinynet_d import tinynet_d
from src.training.trainer_classifier import Trainer

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader

d:\anaconda3\envs\pytorchgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
batch_size = 256

train_path = r"/kaggle/input/datasets/lewisdo/train-cnn/Scalogram/train_synthetic.pt"
train_loader = DataLoader(CWRUDataset(train_path), batch_size=batch_size, shuffle=True)

val_path = r"/kaggle/input/datasets/lewisdo/train-cnn/Scalogram/val.pt"
val_loader = DataLoader(CWRUDataset(val_path), batch_size=batch_size, shuffle=False)

test_path = r"/kaggle/input/datasets/lewisdo/train-cnn/Scalogram/test.pt"
test_loader = DataLoader(CWRUDataset(test_path), batch_size=batch_size, shuffle=False)

In [ ]:
model = mobilenetv4()

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)

In [ ]:
trainer = Trainer(
    model=model,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer,
    scheduler=scheduler,
    num_classes=4,
    class_names=["Normal", "Outer Race", "Inner Race", "Ball"],
    use_data_parallel=False
)

In [ ]:
history = trainer.fit(train_loader, val_loader, epochs=150)

In [ ]:
trainer.summary()
trainer.plot_all(save_dir="./figures")

In [ ]:
trainer.save_model("best_model.pth", include_history=True)